# Section 2. Pydantic과 구조화 출력

이 섹션에서는 LLM 출력이 “그럴듯한 문장”에서 끝나지 않도록 schema를 만듭니다. Pydantic은 결과가 우리가 정한 필드와 타입을 만족하는지 검사합니다.

핵심은 두 가지입니다.

- validation은 형식 검증입니다. 사실 검증은 아닙니다.
- evidence는 문자열 하나가 아니라 `source_id`와 `quote`를 가진 객체입니다.


## schema 만들기

아래 schema는 문서에서 답을 찾았는지, 한국어 답변이 무엇인지, 어떤 근거 문장을 썼는지 표현합니다.


In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Evidence(BaseModel):
    source_id: str = Field(description="근거가 나온 문서 ID")
    quote: str = Field(description="원문에서 그대로 가져온 근거 문장")

class GroundedAnswer(BaseModel):
    answered: bool
    answer_ko: str
    evidence: list[Evidence]


## LangChain parser로 형식 지시문 만들기

LangChain은 prompt, model call, parser 같은 부품을 연결하기 쉽게 해주는 도구입니다. 여기서는 전체 chain을 만들기보다 `PydanticOutputParser`가 Pydantic schema를 읽고 모델에게 줄 형식 지시문을 만드는 모습을 봅니다.


In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=GroundedAnswer)
print(parser.get_format_instructions())


## API 호출 없이 실패 예시 보기

Pydantic은 타입과 필드가 틀리면 즉시 알려줍니다. 이 셀은 일부러 틀린 결과를 넣어 validation 실패를 확인합니다.


In [ ]:
bad_output = {
    "answered": "yes",
    "answer": "인증과 환경 변수가 필요합니다.",
    "evidence": ["authentication, environment variables"],
}

try:
    GroundedAnswer.model_validate(bad_output)
except ValidationError as exc:
    print(exc)


## 올바른 형식 예시

형식이 맞으면 Pydantic 객체로 변환됩니다.


In [ ]:
good_output = {
    "answered": True,
    "answer_ko": "API onboarding에는 인증, 환경 변수, rate limit 확인, smoke test가 필요합니다.",
    "evidence": [
        {
            "source_id": "api_onboarding",
            "quote": "API onboarding requires authentication, environment variables, awareness of rate limits, and one minimal smoke test before larger experiments.",
        }
    ],
}

answer = GroundedAnswer.model_validate(good_output)
answer


## API key 입력

각 실습 노트북은 독립적으로 실행됩니다. 아래 셀의 따옴표 안에 수업용 OpenAI API key를 붙여넣고 실행하세요.

- key가 들어간 노트북은 그대로 공유하지 않습니다.
- 제출하거나 화면 공유하기 전에는 key 문자열을 지웁니다.
- 이 자료에서는 `.env` 파일을 쓰지 않습니다. 학생이 열어야 할 파일 수를 줄이기 위해 노트북 안에서 직접 입력합니다.


In [ ]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "여기에_수업용_API_KEY를_붙여넣으세요")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

if not OPENAI_API_KEY or OPENAI_API_KEY.startswith("여기에_"):
    raise ValueError("환경변수 OPENAI_API_KEY를 설정하거나 이 셀의 자리에 수업용 API key를 붙여넣은 뒤 다시 실행하세요.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key가 현재 노트북 실행 세션에 설정되었습니다.")


## LLM에게 JSON으로 답하게 하기

모델에게 JSON 형식을 요구한 뒤 Pydantic으로 다시 확인합니다. 출력이 깨지면 schema가 방어막 역할을 합니다.


In [ ]:
NOTES = [
    {
        "note_id": "support_triage",
        "text": "Support triage routes billing requests to the finance queue. Urgent security issues go to the incident channel. Uncertain cases use a fallback path for review.",
    },
    {
        "note_id": "api_onboarding",
        "text": "API onboarding requires authentication, environment variables, awareness of rate limits, and one minimal smoke test before larger experiments.",
    },
    {
        "note_id": "rag_quality",
        "text": "A grounded RAG answer should cite the retrieved source, quote only text that appears in that source, and say not answered when the corpus lacks evidence.",
    },
]


In [ ]:
import json
from openai import OpenAI

client = OpenAI()
source = NOTES[1]
prompt = f"""
다음 문서만 근거로 질문에 답하세요.
반드시 JSON만 출력하세요. 필드는 answered, answer_ko, evidence 입니다.
evidence는 source_id와 quote를 가진 객체 배열입니다.
quote는 원문에 실제로 있는 문장만 사용하세요.

문서 ID: {source['note_id']}
문서: {source['text']}
질문: API onboarding을 시작하기 전에 무엇을 확인해야 하나요?
"""

response = client.responses.create(
    model=OPENAI_MODEL,
    input=prompt,
    max_output_tokens=500,
)
raw = response.output_text.strip()
print(raw)

parsed = GroundedAnswer.model_validate(json.loads(raw))
parsed


## 결과 확인

- `answer_ko`가 있어야 합니다. `answer` 필드가 아닙니다.
- `evidence`는 문자열 배열이 아니라 `{source_id, quote}` 객체 배열입니다.
- quote가 실제 문서에 있는지 사람도 확인해야 합니다.
